In [ ]:
import os
import sys
import time
import json
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
from tqdm.auto import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# TorchVision
import torchvision
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet101, DeepLabV3_ResNet101_Weights
from pycocotools.coco import COCO
from PIL import Image


from pycocotools import mask as maskUtils


# Transformers (для SegFormer)
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerFeatureExtractor
)


In [ ]:
@dataclass
class Config:
    coco_root: str = '../coco2017'
    train_root: str = os.path.join(coco_root, 'train2017')
    val_root: str = os.path.join(coco_root, 'val2017')
    train_ann: str = os.path.join(coco_root, 'annotations', 'instances_train2017.json')
    val_ann: str = os.path.join(coco_root, 'annotations', 'instances_val2017.json')

    batch_size: int = 4  # Зменшено для стабільності
    num_epochs: int = 10
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    num_workers: int = 0

    num_classes: int = 81
    img_size: int = 512
    seed: int = 42
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint_dir: str = './checkpoints'
    save_every: int = 2  # Зберігати кожні 2 епохи

cfg = Config()
os.makedirs(cfg.checkpoint_dir, exist_ok=True)
print("\n" + "="*80)
print("КОНФІГУРАЦІЯ ПРОЕКТУ")
print("="*80)
print(f"Dataset: COCO 2017")
print(f"Device: {cfg.device}")
print(f"Batch size: {cfg.batch_size}")
print(f"Image size: {cfg.img_size}")
print(f"Number of classes: {cfg.num_classes}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print("="*80 + "\n")

In [ ]:
class COCOSemanticDataset(Dataset):
    #TODO: Implement the dataset class

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])


In [ ]:
print("Завантаження datasets...")
train_dataset = COCOSemanticDataset(
    img_root=cfg.train_root,
    ann_file=cfg.train_ann,
    transform=train_transform,
    image_size=(cfg.img_size, cfg.img_size)
)

val_dataset = COCOSemanticDataset(
    img_root=cfg.val_root,
    ann_file=cfg.val_ann,
    transform=val_transform,
    image_size=(cfg.img_size, cfg.img_size)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True
)

print(f"Train dataset: {len(train_dataset)} зображень")
print(f"Validation dataset: {len(val_dataset)} зображень")
print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}\n")

In [ ]:
print("Ініціалізація DeepLabV3+ моделі...")
deeplab_model = deeplabv3_resnet101(weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1)

# Замінити останній шар для нашої кількості класів
deeplab_model.classifier[4] = nn.Conv2d(256, cfg.num_classes, kernel_size=(1, 1))
deeplab_model.aux_classifier[4] = nn.Conv2d(256, cfg.num_classes, kernel_size=(1, 1))

# Заморозити backbone для feature extraction (спочатку)
for param in deeplab_model.backbone.parameters():
    param.requires_grad = False

# Розморозити класифікатор для fine-tuning
for param in deeplab_model.classifier.parameters():
    param.requires_grad = True
for param in deeplab_model.aux_classifier.parameters():
    param.requires_grad = True

deeplab_model = deeplab_model.to(cfg.device)
print(f"DeepLabV3+ завантажено на {cfg.device}")

In [ ]:
print("\nІніціалізація SegFormer моделі...")
segformer_model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b3-finetuned-ade-512-512",
    num_labels=cfg.num_classes,
    ignore_mismatched_sizes=True  # Дозволити різні розміри для нових класів
)

# Заморозити encoder для feature extraction
for param in segformer_model.segformer.encoder.parameters():
    param.requires_grad = False

# Розморозити decoder для fine-tuning
for param in segformer_model.decode_head.parameters():
    param.requires_grad = True

segformer_model = segformer_model.to(cfg.device)
print(f"SegFormer завантажено на {cfg.device}\n")

In [ ]:
# Зважена CrossEntropyLoss (більша вага для рідкісних класів)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ігноруємо background

# Окремі оптимізатори для кожної моделі
optimizer_deeplab = optim.AdamW(
    filter(lambda p: p.requires_grad, deeplab_model.parameters()),
    lr=cfg.learning_rate,
    weight_decay=cfg.weight_decay
)

optimizer_segformer = optim.AdamW(
    filter(lambda p: p.requires_grad, segformer_model.parameters()),
    lr=cfg.learning_rate,
    weight_decay=cfg.weight_decay
)

# Learning rate schedulers
scheduler_deeplab = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_deeplab, mode='min', factor=0.5, patience=2
)
scheduler_segformer = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_segformer, mode='min', factor=0.5, patience=2
)

# GradScaler для mixed precision training
scaler = torch.amp.GradScaler()